# Mediapipe

In [ ]:
# Códigos basado en el ejemplo proporcionado por MediaPipe

# DETECCIÓN DE LANDMARKS FACIALES
import time
import cv2
import mediapipe as mp
import numpy as np

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

latest = None  

def on_result(result, output_image, timestamp_ms):
    global latest
    latest = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    output_face_blendshapes=True,
    result_callback=on_result,
)

nose_indices = (
    1, 2, 4, 5, 6, 19, 45, 48, 64, 94, 97, 98,
    115, 168, 195, 197, 220, 275, 278, 294, 326, 327, 344, 440
)

flappy_bird = cv2.imread(r"./assets/flappy_bird.png", cv2.IMREAD_UNCHANGED)
if flappy_bird is None:
    raise FileNotFoundError(r"No se pudo cargar ./assets/flappy_bird.png")

cap = cv2.VideoCapture(0)
with FaceLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        landmarker.detect_async(mp_image, int(time.time() * 1000))

        if latest is not None:
            h, w = frame.shape[:2]

            for face in latest.face_landmarks:
                h, w = frame.shape[:2]
                points = np.array(
                    [(face[i].x * w, face[i].y * h) for i in nose_indices],
                    dtype=np.float32,
                )

                # calcular centro de la nariz
                (center_x, center_y), radius = cv2.minEnclosingCircle(points)

                # se ajusta el tamaño del png del flappy bird
                size = max(1, int(2 * radius * 0.9))
                overlay = cv2.resize(flappy_bird, (size, size), interpolation=cv2.INTER_AREA)

                # se calcula la esquina superior izquierda para el png
                # y se respetan los bordes de la imagen
                x, y = int(center_x - size / 2), int(center_y - size / 2)
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(w, x + size), min(h, y + size)
                overlay = overlay[y1 - y:y2 - y, x1 - x:x2 - x]
                roi = frame[y1:y2, x1:x2]

                alpha = overlay[:, :, 3:4].astype(np.float32) / 255.0
                frame[y1:y2, x1:x2] = (
                    alpha * overlay[:, :, :3] + (1 - alpha) * roi
                ).astype(np.uint8)

        cv2.imshow("Face", frame)
        if cv2.waitKey(1) & 0xFF == 27:  # Esc
            break

cap.release()
cv2.destroyAllWindows()

## Clase para la gestión de las tuberias

In [2]:
import random

def overlay_rgba(frame, img, x, y):
    """Pega una imagen BGRA sobre el frame en (x, y), recortando en los bordes."""
    h, w = frame.shape[:2]
    ih, iw = img.shape[:2]
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(w, x + iw), min(h, y + ih)
    if x1 >= x2 or y1 >= y2:
        return  # completamente fuera de pantalla
    crop = img[y1 - y:y2 - y, x1 - x:x2 - x]
    roi = frame[y1:y2, x1:x2]
    alpha = crop[:, :, 3:4].astype(np.float32) / 255.0
    frame[y1:y2, x1:x2] = (alpha * crop[:, :, :3] + (1 - alpha) * roi).astype(np.uint8)


class PipeManager:
    def __init__(self, frame_w, frame_h, pipe_img_path,
                 spawn_interval=2.0,
                 speed=250,
                 pipe_width=80,
                 gap_height=120,
                 margin=80,
                 cap_src_h=64):
        
        self.frame_w = frame_w
        self.frame_h = frame_h
        self.spawn_interval = spawn_interval
        self.speed = speed
        self.base_speed = speed
        self.spacing = speed * spawn_interval
        self.pipe_width = pipe_width
        self.gap_height = gap_height
        self.margin = margin

        # carga de la imagen
        img = cv2.imread(pipe_img_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"No se pudo cargar {pipe_img_path}")
        if img.shape[2] == 3:  # si no tiene alpha, se lo añadimos
            img = cv2.cvtColor(img, cv2.COLOR_BGR2BGRA)

        # recortar el margen transparente
        ys, xs = np.where(img[:, :, 3] > 0)
        img = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

        src_h, src_w = img.shape[:2]
        scale = pipe_width / src_w
        self.cap_h = max(1, int(cap_src_h * scale))

        # cuerpo (parte superior del PNG) y boca (parte inferior)
        self.body_src = img[:src_h - cap_src_h]
        self.cap = cv2.resize(img[src_h - cap_src_h:], (pipe_width, self.cap_h), interpolation=cv2.INTER_AREA)

        self.pipes = []
        self.last_spawn = time.time()
        self.last_update = time.time()

    def make_pipe(self, length):
        """Tubería con la boca abajo y la altura pedida."""
        length = max(1, length)
        body_h = length - self.cap_h
        parts = []
        if body_h > 0:
            parts.append(cv2.resize(self.body_src, (self.pipe_width, body_h),
                                    interpolation=cv2.INTER_LINEAR))
        parts.append(self.cap)
        return np.vstack(parts)[-length:]

    def spawn(self):
        gap_y = random.randint(self.margin,
                               self.frame_h - self.gap_height - self.margin)
        gap_bottom = gap_y + self.gap_height

        top_img = self.make_pipe(gap_y)
        bottom_img = cv2.flip(self.make_pipe(self.frame_h - gap_bottom), 0)

        self.pipes.append({
            "x": float(self.frame_w),
            "gap_y": gap_y,
            "top_img": top_img,
            "bottom_img": bottom_img,
            "passed": False,
        })

    def update(self):
        now = time.time()
        dt = now - self.last_update
        self.last_update = now

        if now - self.last_spawn >= self.spawn_interval:
            self.spawn()
            self.last_spawn = now

        for p in self.pipes:
            p["x"] -= self.speed * dt

        self.pipes = [p for p in self.pipes if p["x"] + self.pipe_width > 0]

    def draw(self, frame):
        for p in self.pipes:
            x = int(p["x"])
            gap_bottom = p["gap_y"] + self.gap_height
            overlay_rgba(frame, p["top_img"], x, 0)
            overlay_rgba(frame, p["bottom_img"], x, gap_bottom)

    def setSpeed(self, newSpeed):
        self.speed = newSpeed
        self.spawn_interval = self.spacing / newSpeed

    def reset(self):
        self.pipes = []
        self.setSpeed(self.base_speed)
        self.last_spawn = time.time()
        self.last_update = time.time()

### Filtro con las tuberias

In [3]:
import time
import cv2
import mediapipe as mp
import numpy as np

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

latest = None  

def on_result(result, output_image, timestamp_ms):
    global latest
    latest = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    output_face_blendshapes=True,
    result_callback=on_result,
)

nose_indices = (
    1, 2, 4, 5, 6, 19, 45, 48, 64, 94, 97, 98,
    115, 168, 195, 197, 220, 275, 278, 294, 326, 327, 344, 440
)

flappy_bird = cv2.imread(r"./assets/flappy_bird.png", cv2.IMREAD_UNCHANGED)
if flappy_bird is None:
    raise FileNotFoundError(r"No se pudo cargar ./assets/flappy_bird.png")

cap = cv2.VideoCapture(0)

ret, frame = cap.read()
if not ret:
    raise RuntimeError("No se pudo leer de la cámara")

frame_h, frame_w = frame.shape[:2]
pipes = PipeManager(frame_w, frame_h, r"./assets/pipe_green.png")

with FaceLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        landmarker.detect_async(mp_image, int(time.time() * 1000))

        pipes.update()
        pipes.draw(frame)

        if latest is not None:
            h, w = frame.shape[:2]

            for face in latest.face_landmarks:
                h, w = frame.shape[:2]
                points = np.array(
                    [(face[i].x * w, face[i].y * h) for i in nose_indices],
                    dtype=np.float32,
                )

                # calcular centro de la nariz
                (center_x, center_y), radius = cv2.minEnclosingCircle(points)

                # se ajusta el tamaño del png del flappy bird
                size = max(1, int(2 * radius * 0.9))
                overlay = cv2.resize(flappy_bird, (size, size), interpolation=cv2.INTER_AREA)

                # se calcula la esquina superior izquierda para el png
                # y se respetan los bordes de la imagen
                x, y = int(center_x - size / 2), int(center_y - size / 2)
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(w, x + size), min(h, y + size)
                overlay = overlay[y1 - y:y2 - y, x1 - x:x2 - x]
                roi = frame[y1:y2, x1:x2]

                alpha = overlay[:, :, 3:4].astype(np.float32) / 255.0
                frame[y1:y2, x1:x2] = (
                    alpha * overlay[:, :, :3] + (1 - alpha) * roi
                ).astype(np.uint8)

        cv2.imshow("Face", frame)
        if cv2.waitKey(1) & 0xFF == 27:  # Esc
            break

cap.release()
cv2.destroyAllWindows()

## Clase para la gestión de partida

In [4]:
class GameManager:
    def __init__(self, pipeManager):
        self.score = 0
        self.bestScore = 0
        self.pipeManager = pipeManager
        self.gameOver = False
        
        self.maxSpeed = 400
        self.speePerPoint = 7

    def passPipes(self, x):

        for pipe in self.pipeManager.pipes:
            if pipe["x"] < x and pipe["passed"] == False:
                pipe["passed"] = True
                self.score += 1
                self.bestScore = max(self.bestScore, self.score)
                self.pipeManager.setSpeed(min(self.maxSpeed, self.pipeManager.base_speed + self.speePerPoint * self.score))

    @staticmethod
    def detectCollision(cx, cy, r, left, top, right, bottom):
        nearest_x = min(max(cx, left), right)
        nearest_y = min(max(cy, top), bottom)
        dx, dy = cx - nearest_x, cy - nearest_y
        return dx * dx + dy * dy < r * r

    def checkCollisions(self, cx, cy, radius):

        for pipe in self.pipeManager.pipes:
            left = pipe["x"]
            right = left + self.pipeManager.pipe_width
            gap_top = pipe["gap_y"]
            gap_bottom = gap_top + self.pipeManager.gap_height

            # tuberia superior
            if self.detectCollision(cx, cy, radius, left, 0, right, gap_top):
                return True
                
            # tuberia inferior
            if self.detectCollision(cx, cy, radius, left, gap_bottom, right, self.pipeManager.frame_h):
    
                return True

        return False
        

    

In [5]:
import time
import cv2
import mediapipe as mp
import numpy as np

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

latest = None  

def on_result(result, output_image, timestamp_ms):
    global latest
    latest = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    output_face_blendshapes=True,
    result_callback=on_result,
)

nose_indices = (
    1, 2, 4, 5, 6, 19, 45, 48, 64, 94, 97, 98,
    115, 168, 195, 197, 220, 275, 278, 294, 326, 327, 344, 440
)

flappy_bird = cv2.imread(r"./assets/flappy_bird.png", cv2.IMREAD_UNCHANGED)
if flappy_bird is None:
    raise FileNotFoundError(r"No se pudo cargar ./assets/flappy_bird.png")

cap = cv2.VideoCapture(0)

ret, frame = cap.read()
if not ret:
    raise RuntimeError("No se pudo leer de la cámara")

frame_h, frame_w = frame.shape[:2]
pipes = PipeManager(frame_w, frame_h, r"./assets/pipe_green.png")
game = GameManager(pipes)

with FaceLandmarker.create_from_options(options) as landmarker:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        cv2.putText(frame, 
                    f"Puntuación: {game.score}",
                    (15,55),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.1,
                    (0, 255, 0))
        
        cv2.putText(frame, 
                    f"Mejor puntuación: {game.bestScore}",
                    (15,30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (0, 0, 255))


        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        landmarker.detect_async(mp_image, int(time.time() * 1000))

        if not game.gameOver:
            pipes.update() 
        
        pipes.draw(frame)

        if latest is not None:
            h, w = frame.shape[:2]

            for face in latest.face_landmarks:
                h, w = frame.shape[:2]
                points = np.array(
                    [(face[i].x * w, face[i].y * h) for i in nose_indices],
                    dtype=np.float32,
                )

                # calcular centro de la nariz
                (center_x, center_y), radius = cv2.minEnclosingCircle(points)

                # se ajusta el tamaño del png del flappy bird
                size = max(1, int(2 * radius * 0.9))
                overlay = cv2.resize(flappy_bird, (size, size), interpolation=cv2.INTER_AREA)

                # se calcula la esquina superior izquierda para el png
                # y se respetan los bordes de la imagen
                x, y = int(center_x - size / 2), int(center_y - size / 2)

                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(w, x + size), min(h, y + size)
                overlay = overlay[y1 - y:y2 - y, x1 - x:x2 - x]
                roi = frame[y1:y2, x1:x2]

                alpha = overlay[:, :, 3:4].astype(np.float32) / 255.0
                frame[y1:y2, x1:x2] = (
                    alpha * overlay[:, :, :3] + (1 - alpha) * roi
                ).astype(np.uint8)
                
                if game.checkCollisions(center_x, center_y, radius):
                    game.gameOver = True
                else:
                    game.passPipes(x)

                if game.gameOver:
                    
                    cv2.putText(frame, "HAS PERDIDO", (130, 200), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 8, cv2.LINE_AA)
                    cv2.putText(frame, f"Puntos: {game.score}", (230, 260), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 6, cv2.LINE_AA)
                    cv2.putText(frame, "Espacio para reiniciar", (160, 310), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 6, cv2.LINE_AA)


        cv2.imshow("Face", frame)

        key = cv2.waitKey(1) & 0xFF 
        
        if key == 27:
            break
        elif key == 32 and game.gameOver:
            game.pipeManager.reset()
            game.score = 0
            game.gameOver = False

cap.release()
cv2.destroyAllWindows()